In [1]:
# 收入 = 用户 × 行为 × 时间
# GMV驱动
## 本分析目标：拆解GMV周期波动来源（用户数 / 行为频次 / 客单价），并进一步分析老用户结构变化对GMV波动的影响机制
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 

os.getcwd()

'D:\\Python_set\\Python_code\\4_InteractionMARL-Coop-main\\OnlineRetailProject\\purePythonProject\\notebooks'

In [2]:
df = pd.read_csv('../../data/raw/online_retail_II.csv')

In [3]:
# 数据理解（EDA）
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [5]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [6]:
# 数据清洗
# 1. 去重
df = df.drop_duplicates()

# 2. 缺失值（用户无法归因的订单）
df = df.dropna(subset=['Customer ID'])

# 3. 异常值（退款/无效交易）
df = df[df['Price'] > 0]
df = df[df['Quantity'] > 0]
df_refund = df[df['Quantity'] <= 0]

'''
df.drop(df[df['Quantity']<=0].index, inplace = True)
df.drop(df[df['Price']<=0].index, inplace = True)
'''
# 4. 时间格式
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 5. 构造核心指标（后续分析基础）
df['amount'] = df['Quantity'] * df['Price']

In [7]:
# KPI: GMV(month)
df['year_month'] = df['InvoiceDate'].dt.to_period('M')
df_order = df.groupby('Invoice').agg(
    CustomerID = ('Customer ID', 'first'),
    year_month = ('year_month', 'first'),
    amount = ('amount', 'sum'),
    country = ('Country', 'first')
).reset_index()
df_sales_month = df.groupby('year_month')['amount'].sum().sort_index()
df_sales_month

year_month
2009-12     683504.010
2010-01     555802.672
2010-02     504558.956
2010-03     696978.471
2010-04     591982.002
2010-05     597833.380
2010-06     636371.130
2010-07     589736.170
2010-08     602224.600
2010-09     829013.951
2010-10    1033112.010
2010-11    1166460.022
2010-12     570422.730
2011-01     568101.310
2011-02     446084.920
2011-03     594081.760
2011-04     468374.331
2011-05     677355.150
2011-06     660046.050
2011-07     598962.901
2011-08     644051.040
2011-09     950690.202
2011-10    1035642.450
2011-11    1156205.610
2011-12     517208.440
Freq: M, Name: amount, dtype: float64

In [8]:
'''以KPI为切入点，观测常用KPI中的GMV，可以发现GMV呈现明显周期性波动，9–11月为高峰区间（疑似季节性/促销影响）。'''

'以KPI为切入点，观测常用KPI中的GMV，可以发现GMV呈现明显周期性波动，9–11月为高峰区间（疑似季节性/促销影响）。'

In [9]:
# GMV结构拆解（用户/行为/价格）
df_sales_indicators_month = (
    df_order.groupby('year_month')
    .agg(
        GMV_m=('amount', 'sum'),
        num_orders_m = ('Invoice', 'nunique' ),
        num_users_m = ('CustomerID', 'nunique' )
    ).assign(
        APO_m = lambda x : x['num_orders_m'] / x['num_users_m'],
        AOV_m = lambda x : x['GMV_m']/x['num_orders_m'],
        GMV_pc_m = lambda x: x['GMV_m']/x['num_users_m']
    ).sort_index()
)
df_sales_indicators_month

,GMV_m,num_orders_m,num_users_m,APO_m,AOV_m,GMV_pc_m
year_month,,,,,,
2009-12,683504.010,1512,955,1.583246,452.052917,715.711005
2010-01,555802.672,1011,720,1.404167,549.755363,771.948156
2010-02,504558.956,1104,772,1.430052,457.028040,653.573777
2010-03,696978.471,1524,1057,1.441816,457.334955,659.393066
2010-04,591982.002,1329,942,1.410828,445.434163,628.431000
2010-05,597833.380,1377,966,1.425466,434.156412,618.875135
2010-06,636371.130,1497,1041,1.438040,425.097615,611.307522
2010-07,589736.170,1381,928,1.488147,427.035605,635.491563
2010-08,602224.600,1293,911,1.419319,465.757618,661.058836


In [10]:
df1 = df_sales_indicators_month

corr_users = df1.GMV_m.corr(df1.num_users_m)
corr_aov   = df1.GMV_m.corr(df1.AOV_m)
corr_apo   = df1.GMV_m.corr(df1.APO_m)

# 打印输出
print(f"GMV与用户数相关系数：{corr_users:.3f}")
print(f"GMV与客单价相关系数：{corr_aov:.3f}")
print(f"GMV与人均订单相关系数：{corr_apo:.3f}")

GMV与用户数相关系数：0.960
GMV与客单价相关系数：0.037
GMV与人均订单相关系数：0.528


In [11]:
'''
通过指标结构性拆解为用户数、APO、AOV，利用相关性分析发现GMV与用户数高度相关（0.96），与人均订单中等相关（0.53），与客单价几乎无关（0.04），说明GMV主要由用户规模扩张驱动，价格因素影响极弱，行为因素存在一定辅助作用但不构成主导。
'''

'\n通过指标结构性拆解为用户数、APO、AOV，利用相关性分析发现GMV与用户数高度相关（0.96），与人均订单中等相关（0.53），与客单价几乎无关（0.04），说明GMV主要由用户规模扩张驱动，价格因素影响极弱，行为因素存在一定辅助作用但不构成主导。\n'

In [12]:
# 新老用户划分（cohort分析）
df_order['cohort'] = df_order.groupby('CustomerID')['year_month'].transform('min')

df_customers_month = (
    df_order
    .assign(is_new = lambda x: (x.cohort == x.year_month).astype(int))
    .groupby(['year_month', 'CustomerID'], as_index=False)
    .agg(
        is_new=('is_new', 'max'),
        amount_cm = ('amount', 'sum'),
        country_cm = ('country', 'first')
    )
)
df_customers_month

,year_month,CustomerID,is_new,amount_cm,country_cm
0,2009-12,12346.0,1,113.50,United Kingdom
1,2009-12,12358.0,1,1429.83,Austria
2,2009-12,12359.0,1,838.89,Cyprus
3,2009-12,12362.0,1,130.00,Belgium
4,2009-12,12417.0,1,317.60,Belgium
...,...,...,...,...,...
25590,2011-12,18245.0,0,894.25,United Kingdom
25591,2011-12,18272.0,0,367.88,United Kingdom
25592,2011-12,18273.0,0,51.00,United Kingdom
25593,2011-12,18282.0,0,77.84,United Kingdom


In [13]:
# 新老用户规模变化
df_customers_NO_month = (
    df_customers_month
    .groupby('year_month')
    .agg(
        num_news = ('is_new', 'count'),
        num_olds = ('is_new', lambda x : (x==0).sum())
    ).sort_index()
)
df_customers_NO_month

,num_news,num_olds
year_month,,
2009-12,955,0
2010-01,720,337
2010-02,772,398
2010-03,1057,614
2010-04,942,648
2010-05,966,712
2010-06,1041,771
2010-07,928,742
2010-08,911,749


In [14]:
'''以订单初次在本数据集中出现的用户定位新用户，将用户分层为新老用户。# 新老用户规模随GMV同步变化，说明用户活跃规模与GMV存在同步关系，但仅基于人数无法解释GMV结构变化'''

'以订单初次在本数据集中出现的用户定位新用户，将用户分层为新老用户。# 新老用户规模随GMV同步变化，说明用户活跃规模与GMV存在同步关系，但仅基于人数无法解释GMV结构变化'

In [15]:
df_customers_NO_GMV_month = (
    df_customers_month
    .groupby(['year_month', 'is_new'])['amount_cm']
    .sum()
    .unstack(fill_value=0)
    .rename(columns={0: 'GMV_old', 1: 'GMV_new'})
    .rename_axis(columns=None)
    .sort_index()
)
df_customers_NO_GMV_month

,GMV_old,GMV_new
year_month,,
2009-12,0.000,683504.010
2010-01,394723.981,161078.691
2010-02,334916.922,169642.034
2010-03,462793.210,234185.261
2010-04,467716.711,124265.291
2010-05,487426.250,110407.130
2010-06,505326.770,131044.360
2010-07,514688.520,75047.650
2010-08,541561.300,60663.300


In [16]:
'''进一步看新老用户GMV变动，老用户GMV与整体趋势一致性更高，是主要贡献来源，新用户波动较弱，对周期影响有限'''

'进一步看新老用户GMV变动，老用户GMV与整体趋势一致性更高，是主要贡献来源，新用户波动较弱，对周期影响有限'

In [17]:
old_user = df_customers_month[df_customers_month['is_new']==0]
old_user

,year_month,CustomerID,is_new,amount_cm,country_cm
955,2010-01,12346.0,0,90.00,United Kingdom
959,2010-01,12417.0,0,404.55,Belgium
961,2010-01,12437.0,0,609.78,France
962,2010-01,12439.0,0,76.52,Cyprus
966,2010-01,12471.0,0,1021.44,Germany
...,...,...,...,...,...
25590,2011-12,18245.0,0,894.25,United Kingdom
25591,2011-12,18272.0,0,367.88,United Kingdom
25592,2011-12,18273.0,0,51.00,United Kingdom
25593,2011-12,18282.0,0,77.84,United Kingdom


In [18]:
old_user.groupby('year_month')['amount_cm'].mean() # 月度均值（行为稳定性观察）

year_month
2010-01    1171.287777
2010-02     841.499804
2010-03     753.734870
2010-04     721.785048
2010-05     684.587430
2010-06     655.417341
2010-07     693.650296
2010-08     723.045794
2010-09     742.091742
2010-10     757.004589
2010-11     790.877466
2010-12     671.327392
2011-01     771.960254
2011-02     634.677760
2011-03     649.155736
2011-04     569.720455
2011-05     661.362455
2011-06     652.222956
2011-07     655.361570
2011-08     722.628311
2011-09     793.219157
2011-10     811.624864
2011-11     730.154902
2011-12     840.518024
Freq: M, Name: amount_cm, dtype: float64

In [19]:
old_user.groupby('year_month').apply(
    lambda x: x.groupby('CustomerID')['amount_cm'].sum().sort_values(ascending=False).head(int(len(x)*0.1)).sum()/x['amount_cm'].sum(),
    include_groups = False
) # Top用户贡献（结构集中度）

year_month
2010-01    0.644032
2010-02    0.530322
2010-03    0.527784
2010-04    0.481464
2010-05    0.469518
2010-06    0.483359
2010-07    0.502958
2010-08    0.502159
2010-09    0.487608
2010-10    0.469864
2010-11    0.490576
2010-12    0.492266
2011-01    0.565895
2011-02    0.471424
2011-03    0.485161
2011-04    0.441093
2011-05    0.471475
2011-06    0.520510
2011-07    0.493442
2011-08    0.518283
2011-09    0.525419
2011-10    0.523415
2011-11    0.486106
2011-12    0.639767
Freq: M, dtype: float64

In [20]:
'''# top10%用户贡献在0.44–0.64之间波动，说明存在一定结构波动，但属于“贡献集中度变化”，不等价于结构崩溃。'''

'# top10%用户贡献在0.44–0.64之间波动，说明存在一定结构波动，但属于“贡献集中度变化”，不等价于结构崩溃。'

In [21]:
# 老用户长尾结构验证
old_user_gmv = old_user.groupby('CustomerID')['amount_cm'].sum()
old_user_gmv.sort_values(ascending=False)

CustomerID
18102.0    539981.30
14646.0    513906.05
14156.0    302386.40
14911.0    284301.57
17450.0    204942.87
             ...    
17252.0        11.90
14610.0        10.90
16498.0         9.75
15884.0         7.90
16454.0         6.90
Name: amount_cm, Length: 4100, dtype: float64

In [22]:
old_user_gmv.sort_values(ascending=False).head(10).sum()/old_user_gmv.sum()

np.float64(0.1838978748265129)

In [23]:
old_user_gmv.quantile(0.9)/old_user_gmv.quantile(0.5)

np.float64(6.18997297138492)

In [24]:
'''老用户呈现明显长尾结构（头部集中 + 长尾分布）老用户内部差异大，考虑RFM模型中一维作为用户分层依据'''

'老用户呈现明显长尾结构（头部集中 + 长尾分布）老用户内部差异大，考虑RFM模型中一维作为用户分层依据'

In [25]:
df_order

,Invoice,CustomerID,year_month,amount,country,cohort
0,489434,13085.0,2009-12,505.30,United Kingdom,2009-12
1,489435,13085.0,2009-12,145.80,United Kingdom,2009-12
2,489436,13078.0,2009-12,630.33,United Kingdom,2009-12
3,489437,15362.0,2009-12,310.75,United Kingdom,2009-12
4,489438,18102.0,2009-12,2286.24,United Kingdom,2009-12
...,...,...,...,...,...,...
36964,581583,13777.0,2011-12,124.60,United Kingdom,2009-12
36965,581584,13777.0,2011-12,140.64,United Kingdom,2009-12
36966,581585,15804.0,2011-12,329.05,United Kingdom,2011-05
36967,581586,13113.0,2011-12,339.20,United Kingdom,2010-03


In [26]:
df_order_old_user = df_order[df_order['CustomerID'].isin(old_user.CustomerID)]
df_order_old_user

,Invoice,CustomerID,year_month,amount,country,cohort
0,489434,13085.0,2009-12,505.30,United Kingdom,2009-12
1,489435,13085.0,2009-12,145.80,United Kingdom,2009-12
2,489436,13078.0,2009-12,630.33,United Kingdom,2009-12
3,489437,15362.0,2009-12,310.75,United Kingdom,2009-12
4,489438,18102.0,2009-12,2286.24,United Kingdom,2009-12
...,...,...,...,...,...,...
36964,581583,13777.0,2011-12,124.60,United Kingdom,2009-12
36965,581584,13777.0,2011-12,140.64,United Kingdom,2009-12
36966,581585,15804.0,2011-12,329.05,United Kingdom,2011-05
36967,581586,13113.0,2011-12,339.20,United Kingdom,2010-03


In [27]:
# RFM频次分析（行为维度）
fre_old_user = df_order_old_user.groupby(['year_month', 'CustomerID']).agg(
    frequence=('Invoice', 'count'),
    gmv = ('amount', 'sum')
).reset_index()
fre_old_user

,year_month,CustomerID,frequence,gmv
0,2009-12,12346.0,5,113.50
1,2009-12,12358.0,1,1429.83
2,2009-12,12359.0,2,838.89
3,2009-12,12362.0,1,130.00
4,2009-12,12417.0,2,317.60
...,...,...,...,...
23812,2011-12,18245.0,1,894.25
23813,2011-12,18272.0,1,367.88
23814,2011-12,18273.0,1,51.00
23815,2011-12,18282.0,1,77.84


In [28]:
# 基准窗口用户分层（固定标签）
df2 = fre_old_user.query("year_month>='2009-12' and year_month<='2010-05'").groupby('CustomerID', as_index=False)['frequence'].sum().sort_values('frequence', ascending=False)

In [29]:
df2

,CustomerID,frequence
993,14911.0,71
490,13694.0,69
1150,15311.0,69
2140,17850.0,61
2135,17841.0,47
...,...,...
16,12388.0,1
14,12378.0,1
13,12377.0,1
12,12374.0,1


In [30]:
q50 = df2['frequence'].quantile(0.5)
q80 = df2['frequence'].quantile(0.8)
print(q50, q80)

2.0 4.0


In [31]:
df2['segment'] = np.select(
    [
        df2['frequence'] <= q50,
        df2['frequence'] <= q80
    ],
    [
        'low',
        'mid'
    ],
    default='high'
)
df2

,CustomerID,frequence,segment
993,14911.0,71,high
490,13694.0,69,high
1150,15311.0,69,high
2140,17850.0,61,high
2135,17841.0,47,high
...,...,...,...
16,12388.0,1,low
14,12378.0,1,low
13,12377.0,1,low
12,12374.0,1,low


In [32]:
# 固定用户分层后的时间追踪
df_customers_observed = fre_old_user[fre_old_user['CustomerID'].isin(df2.CustomerID)].merge(df2, on='CustomerID', how='left', suffixes=['_customer', '_6month']).drop('frequence_6month', axis=1)

In [33]:
df_customers_observed

,year_month,CustomerID,frequence_customer,gmv,segment
0,2009-12,12346.0,5,113.50,high
1,2009-12,12358.0,1,1429.83,low
2,2009-12,12359.0,2,838.89,mid
3,2009-12,12362.0,1,130.00,low
4,2009-12,12417.0,2,317.60,high
...,...,...,...,...,...
16934,2011-12,18225.0,1,330.28,mid
16935,2011-12,18245.0,1,894.25,high
16936,2011-12,18272.0,1,367.88,low
16937,2011-12,18273.0,1,51.00,low


In [34]:
'''切前六个月的老用户作为基准月，取[0,0.5],[0.5,0.8],[0.8, 1]的频率分位数划分为高中低三类用户，固定标签，观测这批用户的数据变化'''

'切前六个月的老用户作为基准月，取[0,0.5],[0.5,0.8],[0.8, 1]的频率分位数划分为高中低三类用户，固定标签，观测这批用户的数据变化'

In [35]:
df3 = df_customers_observed.query("year_month>'2010-05'").groupby(['year_month', 'segment']).agg(
    fre_segcustomer = ('frequence_customer', 'sum'),
    gmv_segcustomer = ('gmv', 'sum'),
    gmv_segcustomer_pc = ('gmv', 'mean')
).reset_index()
df3

,year_month,segment,fre_segcustomer,gmv_segcustomer,gmv_segcustomer_pc
0,2010-06,high,554,299772.190,1189.572183
1,2010-06,low,381,120224.060,373.366646
2,2010-06,mid,252,85330.520,433.149848
3,2010-07,high,527,295975.440,1203.152195
4,2010-07,low,331,117060.990,425.676327
5,2010-07,mid,228,85613.320,492.030575
6,2010-08,high,469,293082.390,1257.864335
7,2010-08,low,300,114266.740,441.184324
8,2010-08,mid,237,103281.370,583.510565
9,2010-09,high,573,344865.570,1401.892561


In [36]:
# 结构占比分析（贡献结构）
gmv_sum = df3.groupby('year_month')['gmv_segcustomer'].transform('sum')
df3['gmv_ratio'] = df3['gmv_segcustomer'] / gmv_sum.round(2)

In [37]:
df3

,year_month,segment,fre_segcustomer,gmv_segcustomer,gmv_segcustomer_pc,gmv_ratio
0,2010-06,high,554,299772.190,1189.572183,0.593224
1,2010-06,low,381,120224.060,373.366646,0.237913
2,2010-06,mid,252,85330.520,433.149848,0.168862
3,2010-07,high,527,295975.440,1203.152195,0.593554
4,2010-07,low,331,117060.990,425.676327,0.234756
5,2010-07,mid,228,85613.320,492.030575,0.171690
6,2010-08,high,469,293082.390,1257.864335,0.573962
7,2010-08,low,300,114266.740,441.184324,0.223776
8,2010-08,mid,237,103281.370,583.510565,0.202262
9,2010-09,high,573,344865.570,1401.892561,0.559758


In [38]:
'''高频用户贡献占比约50%–70%，是GMV核心驱动群体，GMV峰值主要由订单量（频次）驱动，而非客单价提升，属于典型“行为驱动型增长结构”）'''

'高频用户贡献占比约50%–70%，是GMV核心驱动群体，GMV峰值主要由订单量（频次）驱动，而非客单价提升，属于典型“行为驱动型增长结构”）'